In [ ]:
import torch
print(torch.__version__)
!nvcc --version
!python --version

In [ ]:
!git clone https://github.com/okcze/GFlowNet-CombOpt.git

In [ ]:
%%capture
!pip install hydra-core omegaconf submitit hydra-submitit-launcher einops ipdb
if torch.cuda.is_available():
    # !pip install  dgl -f https://data.dgl.ai/wheels/torch-2.4/cu121/repo.html
    !pip install --pre dgl -f https://data.dgl.ai/wheels-test/torch-2.4/cu121/repo.html
    # !pip install  --force-reinstall dglgo -f https://data.dgl.ai/wheels-test/repo.html
else:
    !pip install dgl==1.1.3
!pip install --force-reinstall -v "networkx==2.8.8"
# !pip install --upgrade hydra-core

In [ ]:
# Imports after restart
import os
import torch
import numpy as np
import datetime
import shutil
import zipfile
from google.colab import drive
import itertools
import dgl
import hydra

In [ ]:
print(torch.__version__)
print(dgl.__version__)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
drive.mount('/content/drive')

In [ ]:
%cd /content/GFlowNet-CombOpt

In [ ]:
%%capture
! python ./data/rbgraph_generator.py --num_graph 10 --graph_type small --save_dir ./data/rb200-300/train
# ! python ./data/rbgraph_generator.py --num_graph 50 --graph_type small --save_dir ./data/rb200-300/test

In [ ]:
with zipfile.ZipFile('/content/drive/MyDrive/master_data/test_500_mis.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/test_500_mis')

In [ ]:
os.makedirs('/content/GFlowNet-CombOpt/data/rb200-300/', exist_ok=True)
! cp -r /content/test_500_mis/content/GFlowNet-CombOpt/data/rb200-300/test /content/GFlowNet-CombOpt/data/rb200-300/test

In [ ]:
# Create directories for sampling outputs
!mkdir ./states
!mkdir ./logits

In [ ]:
### Set the config
task = 'mis'
ref_alg = 'mis_greedy'
dates = ['2025-02-15']
NUM_REPEAT = 1

param_set = [
    'frac_0_50_boost_50',
    'frac_0_00_boost_1',
]

In [ ]:
# Fetch trained models
for date in dates:
    for param in param_set:
        if os.path.isdir(f'/content/drive/MyDrive/master_data/{date}/{task}/{param}'):
            !cp -r /content/drive/MyDrive/master_data/{date}/{task}/{param}/alg.pt /content/{param}.pt
            # !cp -r /content/drive/MyDrive/master_data/{date}/{task}/{param}/finetune/{param}/alg.pt /content/{param}.pt

In [ ]:
# Greedy algorithm inference
if torch.cuda.is_available():
    ! python -m gflownet.sample task={task} ref_alg={ref_alg} epochs=1 trainsize=10
else:
    ! python -m gflownet.sample task={task} d=-1 ref_alg={ref_alg} epochs=1 trainsize=10

In [ ]:
# Fixed start GFN inference
for param in param_set:
    if torch.cuda.is_available():
        ! python -m gflownet.sample_fixed_start task={task} ref_alg={ref_alg} fixed_steps={3} epochs=1 trainsize=10 alg_load_path=/content/{param}.pt
    else:
        ! python -m gflownet.sample_fixed_start task={task} d=-1 ref_alg={ref_alg} fixed_steps={3} epochs=1 trainsize=10 alg_load_path=/content/{param}.pt

In [ ]:
# # GFN versions inference
# for param in param_set:
#     if torch.cuda.is_available():
#         ! python -m gflownet.sample task={task} epochs={NUM_REPEAT} trainsize=10 testsize=50 alg_load_path=/content/{param}.pt
#     else:
#         ! python -m gflownet.sample d=-1 task={task} epochs={NUM_REPEAT} trainsize=10 testsize=50 alg_load_path=/content/{param}.pt

In [ ]:
# Logits
if torch.cuda.is_available():
    ! python -m gflownet.logits task={task} epochs=1 trainsize=10 alg_load_path=/content/frac_0_00_boost_1.pt
else:
    ! python -m gflownet.logits d=-1 task={task} epochs=1 trainsize=10 alg_load_path=/content/frac_0_00_boost_1.pt

In [ ]:
class MetricsScorer:
    def __init__(self, task):
        self.task = task

    def score_traj(self, traj):
        s = traj[-1]
        if self.task == 'mds':
            return -(s == 1).sum()
        elif self.task == 'mis':
            return np.sum(s)
        elif self.task == 'maxcut':
            return np.sum(s)
        else:
            raise NotImplementedError

    def dropdown(self, baseline_score, ref_score):
        """Measures relative loss comparing to vanilla GFN solution"""
        return (ref_score - baseline_score)/np.abs(baseline_score)

    def likelihood(self, logits, states):
        """Measures likelihood of arbitrary trajectory under vanilla GFN policy"""
        lh = 0  # Start with 0 since we're summing log probabilities
        for i in range(len(logits)-1):
            differing_indices = np.where(states[i] != states[i+1])[0]
            for idx in differing_indices:
                if states[i+1][idx] == 1:
                    lh += logits[i][idx]  # Sum the log probabilities (logits)
        return lh

    def intersection(self, t_greedy, t2):
        """Measures intersection between two solutions"""
        s1, s2 = t_greedy[-1], t2[-1]
        s1_nodes, s2_nodes = np.where(s1 == 1)[0], np.where(s2 == 1)[0]
        return len(set(s1_nodes) & set(s2_nodes))/len(s1_nodes)

    def get_action(self, s, s_next):
        curr = set(np.where(s==1)[0].tolist())
        next = set(np.where(s_next==1)[0].tolist())
        return next.difference(curr)

    def similarity(self, t1, t2):
        """Measures similarity between two trajectories"""
        min_len = min(len(t1), len(t2))
        max_len = max(len(t1), len(t2))
        sim = []
        for step in range(min_len-1):
            a1, a2 = self.get_action(t1[step], t1[step+1]), self.get_action(t2[step], t2[step+1])
            sim.append(a1 == a2)
        for step in range(min_len, max_len):
            sim.append(False)
        return sim

    def get_max_logit_action(self, logits):
        """Finds the logits maximized actions"""
        max_trajectory = []
        for i in range(len(logits)-1):
            maximized_action = int(np.argmax(logits[i]))
            max_trajectory.append(maximized_action)
        return max_trajectory

    def actions_traj(self, t):
        """Extracts actions from trajectory"""
        t_len = len(t)
        actions = []
        for step in range(t_len-1):
            a = self.get_action(t[step], t[step+1])
            if a:
                actions.append(a.pop())
            else:
                actions.append(None)
        return actions

In [ ]:
# Likelihoods and Scores
scorer = MetricsScorer(task)
algorithms = os.listdir('/content/GFlowNet-CombOpt/states')

lhs_scores = []
for it in range(NUM_REPEAT):

    results_to_gfn = {}
    for alg in algorithms:

        scores = []
        lhs = []
        max_actions = []
        sampling_actions = []

        states = os.listdir(f'/content/GFlowNet-CombOpt/states/{alg}/{it}')
        for graph in states:

            gfn_traj = np.load(f'./states/frac_0_00_boost_1/{it}/{graph}')
            ref_traj = np.load(f'./states/{alg}/{it}/{graph}')
            logits = np.load(f'./logits/{alg}/{graph}')
            logits = logits.squeeze(axis=1)

            lhs.append(scorer.likelihood(logits, ref_traj))
            scores.append(scorer.score_traj(ref_traj))
            max_actions.append(scorer.get_max_logit_action(logits))
            sampling_actions.append(scorer.actions_traj(ref_traj))

        scores = np.array(scores)
        lhs = np.array(lhs)

        results_to_gfn[alg] = {
            "likelihood": lhs,
            "score": scores,
            "max_actions": max_actions,
            "sampling_actions": sampling_actions,
        }
    lhs_scores.append(results_to_gfn)

In [ ]:
graph_id = 20
print(f"Greedy: {lhs_scores[0]['MISGreedy']['sampling_actions'][graph_id]}")
print(f"Greedy: {lhs_scores[0]['frac_0_50_boost_50']['max_actions'][graph_id]}")
print(f"Greedy: {lhs_scores[0]['frac_0_50_boost_50']['sampling_actions'][graph_id]}")

In [ ]:
all_data = []

for alg in algorithms:
    for it in range(NUM_REPEAT):
        for x in lhs_scores[it][alg]["score"]:
            all_data.append({"Algorithm": alg, "Score": x, "Iteration": it})
        for x in lhs_scores[it][alg]["likelihood"]:
            all_data.append({"Algorithm": alg, "Likelihood": x, "Iteration": it})

df = pd.DataFrame(all_data)

In [ ]:
# Compare to the reference algorithm
scorer = MetricsScorer(task)
algorithms = os.listdir('/content/GFlowNet-CombOpt/states')

reg_results = []
for it in range(NUM_REPEAT):

    results_iter = {}
    for alg in algorithms:

        similarities = []
        intersections = []

        states = os.listdir(f'/content/GFlowNet-CombOpt/states/{alg}/0')
        for graph in states:

            greedy_traj = np.load(f'./states/MISGreedy/0/{graph}')
            ref_traj = np.load(f'./states/{alg}/{it}/{graph}')

            similarities.append(scorer.similarity(greedy_traj, ref_traj))
            intersections.append(scorer.intersection(greedy_traj, ref_traj))

        results_iter[alg] = {
            "similarity": similarities,
            "intersection": intersections,
        }
    reg_results.append(results_iter)

In [ ]:
reg_data = []

for alg in algorithms:
    for it in range(NUM_REPEAT):
        for x in reg_results[it][alg]["similarity"]:
            reg_data.append({"Algorithm": alg, "Similarity": x, "Iteration": it})
        for x in reg_results[it][alg]["intersection"]:
            reg_data.append({"Algorithm": alg, "Intersection": x, "Iteration": it})

df_reg = pd.DataFrame(reg_data)

In [ ]:
# Rename frac_0_00_boost_1 to canonical GFN
df.loc[df.Algorithm == 'frac_0_00_boost_1', 'Algorithm'] = 'Canonical GFN'
df_reg.loc[df_reg.Algorithm == 'frac_0_00_boost_1', 'Algorithm'] = 'Canonical GFN'
# Rename frac_0_50_boost_50 to 'f=0.5 boost=50'
df.loc[df.Algorithm == 'frac_0_50_boost_50', 'Algorithm'] = 'Regularized GFN'
df_reg.loc[df_reg.Algorithm == 'frac_0_50_boost_50', 'Algorithm'] = 'Regularized GFN'

In [ ]:
algorithms = ["Canonical GFN", "Regularized GFN", "MISGreedy"]
hue_order = algorithms

Scores

In [ ]:
df[df.Iteration == 0].groupby(["Algorithm"])["Score"].agg(["median", "mean", "std", "min", "max"]).sort_values(by="mean", ascending=False).to_latex(
    f'/content/{task}_scores.tex',
    float_format="%.2f",
    caption=f"Scores summary on {task} task.",
    label=f"tab:scores_{task}",
    column_format="l|ccccc",
    bold_rows=True,
)
df[df.Iteration == 0].groupby(["Algorithm"])["Score"].agg(["median", "mean", "std", "min", "max"]).sort_values(by="mean", ascending=False)

In [ ]:
df_rel = df[df.Iteration == 0].groupby(["Algorithm"])["Score"].agg(["median", "mean", "std", "min", "max"]).sort_values(by="mean", ascending=False).copy(deep=True)

canonical_gfn = df_rel.loc['Canonical GFN']

# Calculate relative difference for each metric
for metric in ['median', 'mean', 'std', 'min', 'max']:
    df_rel[f'{metric}_diff'] = (df_rel[metric] - canonical_gfn[metric]) / canonical_gfn[metric]

# Display the result
df_rel.iloc[:, -5:].to_csv(f"/content/{task}_scores_diff.csv")

Scores histogram

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(18, 8), gridspec_kw={'width_ratios': [3, 2]})

palette = sns.color_palette("colorblind", n_colors=len(hue_order))

# Plot grouped histogram on the left
sns.histplot(
    data=df[df.Iteration == 0],
    x="Score",
    hue="Algorithm",
    discrete=True,
    bins=5,
    palette=palette,
    hue_order=hue_order,
    common_norm=False,
    stat="probability",
    multiple="dodge",  # <-- Changed from "layer" to "dodge" for grouped histogram
    alpha=0.8,
    ax=ax[0],
    shrink=0.8
)
ax[0].set_title("Scores Distributions Histogram (Grouped)")
ax[0].set_xlabel("Score")
sns.move_legend(ax[0], "upper left")

# Plot boxplot on the right
sns.boxplot(
    data=df[df.Iteration == 0],
    y="Score",
    hue="Algorithm",
    hue_order=hue_order,
    palette=palette,
    gap=0.2,
    showfliers=True,
    legend=False,
    ax=ax[1]
)
ax[1].set_title("Scores Distributions Boxplots")
ax[1].set_xlabel("Model")

# Move legend to combined position
sns.move_legend(ax[0], "lower center", bbox_to_anchor=(1.05, -0.2), ncol=3)

# Save figure with proper bounding box
plt.savefig(f"/content/{task}_scores_hist_box.png", dpi=300, bbox_inches="tight")

# Adjust layout and show plot
plt.show()


In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(18, 8), gridspec_kw={'width_ratios': [3, 2]})

palette = sns.color_palette("colorblind", n_colors=len(hue_order))

# Plot histogram on the left
sns.histplot(
    data=df[df.Iteration == 0],
    x="Score",
    hue="Algorithm",
    discrete=True,
    bins=20,
    palette=palette,
    hue_order=hue_order,
    common_norm=False,
    stat="probability",
    multiple="layer",
    alpha=0.7,
    ax=ax[0]
)
ax[0].set_title("Scores Distributions Histogram")
ax[0].set_xlabel("Score")
sns.move_legend(ax[0], "upper left")

# Plot boxplot on the right
sns.boxplot(
    data=df[df.Iteration == 0],
    y="Score",
    hue="Algorithm",
    hue_order = hue_order,
    palette=palette,
    gap=0.2,
    showfliers=True,
    legend=False,
    ax=ax[1]
)
ax[1].set_title("Scores Distributions Boxplots")
ax[1].set_xlabel("Model")
sns.move_legend(ax[0], "lower center", bbox_to_anchor=(1.05, -0.2), ncol=3)

# Save figure with proper bounding box
plt.savefig(f"/content/{task}_scores_hist_box.png", dpi=300, bbox_inches="tight")

# Adjust layout and show plot
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df[df.Iteration == 0], y="Algorithm", x="Score")
plt.title("Scores boxplots")
plt.show()

Comparison of regularized models to vanilla GFN

In [ ]:
df[df.Iteration == 0].groupby(["Algorithm"])["Likelihood"].agg(["median", "mean", "std", "min", "max"]).sort_values(by="mean", ascending=False).to_latex(
    f'/content/{task}_lhs.tex',
    float_format="%.2f",
    caption=f"Likelihoods summary on {task} task.",
    label=f"tab:lhs_{task}",
    column_format="l|ccccc",
    bold_rows=True,
    position="centered",
)
df[df.Iteration == 0].groupby(["Algorithm"])["Likelihood"].agg(["median", "mean", "std", "min", "max"]).sort_values(by="mean", ascending=False)

In [ ]:
df_rel = df[df.Iteration == 0].groupby(["Algorithm"])["Likelihood"].agg(["median", "mean", "std", "min", "max"]).sort_values(by="mean", ascending=False).copy(deep=True)

canonical_gfn = df_rel.loc['Canonical GFN']

# Calculate relative difference for each metric
for metric in ['median', 'mean', 'std', 'min', 'max']:
    df_rel[f'{metric}_diff'] = (canonical_gfn[metric] - df_rel[metric]) / canonical_gfn[metric]

# Display the result
df_rel.iloc[:, -5:].to_csv(f"/content/{task}_lhs_diff.csv")

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(18, 8), gridspec_kw={'width_ratios': [3, 2]})

hue_order = ["Canonical GFN", "Regularized GFN", "MISGreedy"]
palette = sns.color_palette("colorblind", n_colors=len(hue_order))

# Plot histogram on the left
sns.histplot(
    data=df[df.Iteration == 0],
    x="Likelihood",
    hue="Algorithm",
    # discrete=True,
    # bins=20,
    palette=palette,
    hue_order=hue_order,
    common_norm=False,
    stat="probability",
    multiple="layer",
    alpha=0.7,
    ax=ax[0]
)
ax[0].set_title("Likelihoods Distributions Histogram")
ax[0].set_xlabel("Likelihood")
sns.move_legend(ax[0], "upper left")

# Plot boxplot on the right
sns.boxplot(
    data=df[df.Iteration == 0],
    y="Likelihood",
    hue="Algorithm",
    hue_order = hue_order,
    palette=palette,
    gap=0.2,
    showfliers=True,
    legend=False,
    ax=ax[1]
)
ax[1].set_title("Likelihoods Distributions Boxplots")
ax[1].set_xlabel("Model")
sns.move_legend(ax[0], "lower center", bbox_to_anchor=(1.05, -0.2), ncol=3)

# Save figure with proper bounding box
plt.savefig(f"/content/{task}_lhs_hist_box.png", dpi=300, bbox_inches="tight")

# Adjust layout and show plot
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df[df.Iteration == 0], y="Algorithm", x="Likelihood")
plt.title("Likelihood boxplots")
plt.show()

Comparison of GFNs to reference algorithm

In [ ]:
df_reg[df_reg.Iteration == 0].groupby(["Algorithm"])["Intersection"].agg(["median", "mean", "std", "min", "max"]).sort_values(by="mean", ascending=False).to_latex(
    f'/content/{task}_intersections.tex',
    float_format="%.2f",
    caption=f"Intersections summary on {task} task.",
    label=f"tab:intersections_{task}",
    column_format="l|ccccc",
    bold_rows=True,
    position="centered",
)
df_reg[df_reg.Iteration == 0].groupby(["Algorithm"])["Intersection"].agg(["median", "mean", "std", "min", "max"]).sort_values(by="mean", ascending=False)

Intersections of nodes in the solutions (normalized)

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(18, 8), gridspec_kw={'width_ratios': [3, 2]})

hue_order = ["Canonical GFN", "Regularized GFN"]
palette = sns.color_palette("colorblind", n_colors=len(hue_order))

# Plot histogram on the left
sns.histplot(
    data=df_reg[(df_reg.Iteration == 0) & (df_reg.Algorithm != "MISGreedy")],
    x="Intersection",
    hue="Algorithm",
    # discrete=True,
    # bins=18,
    palette=palette,
    hue_order=hue_order,
    # common_norm=False,
    # stat="probability",
    multiple="layer",
    alpha=0.7,
    ax=ax[0]
)
ax[0].set_title("Intersections Distributions Histogram")
ax[0].set_xlabel("Intersection")
sns.move_legend(ax[0], "upper left")

# Plot boxplot on the right
sns.boxplot(
    data=df_reg[(df_reg.Iteration == 0) & (df_reg.Algorithm != "MISGreedy")],
    y="Intersection",
    hue="Algorithm",
    hue_order = hue_order,
    palette=palette,
    gap=0.2,
    showfliers=True,
    legend=False,
    ax=ax[1]
)
ax[1].set_title("Intersections  Distributions Boxplots")
ax[1].set_xlabel("Model")
sns.move_legend(ax[0], "lower center", bbox_to_anchor=(1.05, -0.2), ncol=3)

# Save figure with proper bounding box
plt.savefig(f"/content/{task}_intersections_hist_box.png", dpi=300, bbox_inches="tight")

# Adjust layout and show plot
plt.show()

Similarities of the trajectories

In [ ]:
def fractions(sim_trajectories):
    max_len = max(len(lst) for lst in sim_trajectories)

    fractions = []
    for i in range(max_len):
        values_at_i = [lst[i] for lst in sim_trajectories if len(lst) > i]
        fraction_true = sum(values_at_i) / len(values_at_i)
        fractions.append(fraction_true)
    return fractions

def interseactions_per_graph(df):
    output = [0]*500
    for alg in algorithms:
        x = df[df.Algorithm == alg].Intersection.dropna().values
        # Add values at positions
        for i in range(len(x)):
            output[i] += x[i]
    return np.array(output)/len(algorithms)

In [ ]:
plt.figure(figsize=(10, 6))
for alg in algorithms:
    traj_sim = df_reg[df_reg.Algorithm == alg]['Similarity'].dropna().values
    traj_sim_frac = fractions(traj_sim)
    plt.plot(range(len(traj_sim_frac)), traj_sim_frac, label=alg)
plt.xlabel("Trajectory Step")
plt.ylabel("Regularization Ratio")
plt.legend(algorithms)
plt.title("Regularization Ratio on Test Set Graphs")
plt.savefig(f"/content/{task}_regularization_ratio.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
sns.histplot(df_reg[df_reg.Algorithm == "Regularized GFN"]["Intersection"])
plt.title("Average intersections per graph")
plt.show()

Dropdowns

In [ ]:
# Scores dropdowns
dropdowns = {}
baseline = "frac_0_00_boost_1"
for alg in results_to_gfn.keys():
    dropdowns[alg] = {}
    if alg != baseline:
        dropdowns[alg]["Score"] = scorer.dropdown(results_to_gfn[baseline]["score"], results_to_gfn[alg]["score"])
        dropdowns[alg]["Likelihood"] = scorer.dropdown(results_to_gfn[baseline]["likelihood"], results_to_gfn[alg]["likelihood"])

baseline = "MISGreedy"
reg_results_0 = reg_results[0]
for alg in reg_results_0.keys():
    if alg != baseline:
        # dropdowns[alg]["Regularization Ratio"] = scorer.dropdown(reg_results_0[baseline]["similarity"], reg_results_0[alg]["similarity"])
        dropdowns[alg]["Intersection"] = scorer.dropdown(np.array(reg_results_0[baseline]["intersection"]), np.array(reg_results_0[alg]["intersection"]))

In [ ]:
dd_data = []

for alg in results_to_gfn.keys():
    for x in dropdowns[alg]:
        for y in dropdowns[alg][x]:
            dd_data.append({"Algorithm": alg, "Metric": x, "Dropdown": y})

df_dd = pd.DataFrame(dd_data)

# Rename frac_0_00_boost_1 to canonical GFN
df_dd.loc[df_dd.Algorithm == 'frac_0_00_boost_1', 'Algorithm'] = 'Canonical GFN'
# Rename frac_0_50_boost_50 to 'f=0.5 boost=50'
df_dd.loc[df_dd.Algorithm == 'frac_0_50_boost_50', 'Algorithm'] = 'Regularized GFN'

In [ ]:
df_dd[df_dd.Metric != 'Intersection'].groupby(["Metric", "Algorithm"])["Dropdown"].agg(["median", "mean", "std", "min", "max"]).to_csv(f"/content/{task}_dropdowns.csv")
df_dd[df_dd.Metric != 'Intersection'].groupby(["Metric", "Algorithm"])["Dropdown"].agg(["median", "mean", "std", "min", "max"])

In [ ]:
df_dd[df_dd.Metric == 'Intersection'].groupby(["Metric", "Algorithm"])["Dropdown"].agg(["median", "mean", "std", "min", "max"]).to_csv(f"/content/{task}_dropdowns_intersection.csv")
df_dd[df_dd.Metric == 'Intersection'].groupby(["Metric", "Algorithm"])["Dropdown"].agg(["median", "mean", "std", "min", "max"])

In [ ]:
metric = "Score"
fig, ax = plt.subplots(ncols=2, figsize=(18, 8), gridspec_kw={'width_ratios': [3, 2]})

hue_order = ["Regularized GFN", "MISGreedy"]
palette = sns.color_palette("colorblind", n_colors=len(hue_order))

# Plot histogram on the left
sns.histplot(
    data=df_dd[df_dd.Metric == metric],
    x="Dropdown",
    hue="Algorithm",
    # discrete=True,
    # bins=20,
    palette=palette,
    hue_order=hue_order,
    common_norm=False,
    stat="probability",
    multiple="layer",
    alpha=0.7,
    ax=ax[0]
)
ax[0].set_title("Dropdowns Distributions Histogram")
ax[0].set_xlabel("Dropdown")
sns.move_legend(ax[0], "upper left")

# Plot boxplot on the right
sns.boxplot(
    data=df_dd[df_dd.Metric == metric],
    y="Dropdown",
    hue="Algorithm",
    hue_order = hue_order,
    palette=palette,
    gap=0.2,
    showfliers=True,
    legend=False,
    ax=ax[1]
)
ax[1].set_title("Dropdowns Distributions Boxplots")
ax[1].set_xlabel("Model")
sns.move_legend(ax[0], "lower center", bbox_to_anchor=(1.05, -0.2), ncol=3)

# Save figure with proper bounding box
# plt.savefig(f"/content/{task}_lhs_hist_box.png", dpi=300, bbox_inches="tight")

# Adjust layout and show plot
plt.show()